In [1]:
import os
import pandas as pd
from pathlib import Path
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (PepNet)

This notebook curates the **PepNet** source by assembling single task-specific peptide dataset from heterogeneous inputs. The source provides txt files organized by split (train/test). Here we parse and standardize all inputs, harmonize label conventions, perform duplicate consistency checks per task, and export curated datasets and metadata for downstream analysis.

- **Toxic effect / endpoint:** anti mammalian cells
- **Source:** PepNet
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads multiple txt files** across tasks and splits using a helper loader:
  - assigns `label = 0` for files whose path/name contains `"neg"`,
  - assigns `label = 1` otherwise,
  - stores the originating `source_file` for traceability.
- **Builds task-specific dataset**:
  - `anti mammalian cells`
- **Checks duplicated sequences independently per task**:
  - unique sequences are retained,
  - duplicates with consistent labels are collapsed,
  - sequences with conflicting labels are flagged as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:
  - `processed_anti_mammalian_cells_dataset.csv`
  - `detected_error_sequences.csv`
  - `metadata.json`

In [2]:
name_source = "PepNet"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
def read_multi_files(files):
    dfs = []
    for file in files:
        path = Path(PATH_INPUT) / name_source / file
        df = read_fasta_doc(path)
        df["label"] = (
            df["id"]
            .str.rsplit("|", n=1)
            .str[-1]
            .astype(int)
        )

        df["source_file"] = path.name
        dfs.append(df)

    df_fasta = pd.concat(dfs, ignore_index=True)

    return df_fasta

In [4]:
files = [
    "anti_mammalian_cells_split_train.txt",
    "anti_mammalian_cells_split_valid.txt",
    "test_anti_mammalian_cells.txt",
    "train_anti_mammalian_cells.txt",
]

df = read_multi_files(files)
df.head()

,id,sequence,label,source_file
0,NONFTRAMP00017075|0,YGRRARRRARR,0,anti_mammalian_cells_split_train.txt
1,NONFTRAMP00008210|0,WEEWDREINNYTKLIHELIEESQNQQEENEQELL,0,anti_mammalian_cells_split_train.txt
2,NONFTRAMP00002469|0,LGDARLVITTYWGLHT,0,anti_mammalian_cells_split_train.txt
3,NONFTRAMP00002959|0,MKKGIHPLKRSLDVIMTNGSFVKTIIVSSYIKKNLKLDIDTNKHPC...,0,anti_mammalian_cells_split_train.txt
4,NONFTRAMP00005917|0,MKETAAAKFERQHMDSPDLGTLVPRGSMADIGSTTSNGRQCAGIRP...,0,anti_mammalian_cells_split_train.txt


- Checking duplicates

In [5]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df, group_seq="sequence", sort_key="label")

In [6]:
df_full = pd.concat([df_unique, df_remove_duplicated])

In [7]:
df_errors.shape

(0, 1)

- Working with metada

In [8]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [9]:
dict_metadata.update({
    "number_of_raw_sequences": int(df.shape[0]),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences" : len(df_errors),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'Creative Commons Attribution 4.0',
 'year of publication': 2024,
 'last update date': datetime.datetime(2024, 8, 6, 0, 0),
 'download date': Timestamp('2026-09-02 00:00:00'),
 'file format': 'txt',
 'peptide property': 'anti_mammalian_cells, toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Known not to have any antimicrobial activities',
 'repository or server': 'https://zenodo.org/records/13223516',
 'publication': 'https://www.nature.com/articles/s42003-024-06911-1',
 'number_of_raw_sequences': 39659,
 'number_of_sequences_retained': 22787,
 'number_of_positive_sequences': 3681,
 'number_of_negative_sequences': 19106,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [10]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [11]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_anti_mammalian_cells_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)